# 2.3 로지스틱회귀: 시그모이드에서 PR-AUC까지 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter02_3_logistic_regression.ipynb)

책 본문: [2.3 로지스틱회귀: 시그모이드에서 PR-AUC까지](https://smhanlab.com/book-ml/kor/ml1/chapter02/3.html)

이 노트북은 책 2.3절의 내용을 코드로 재현합니다: (1) 본문 `logistic_gradient_descent` 코드로 실제 불균형 스팸 데이터에 학습시켜 **그래디언트가 `(h-y)x`인지** 확인하고, (2) `sigmoid'(z)=sigmoid(z)(1-sigmoid(z))` 성립을 수치로 검증하고, (3) **임계값을 움직이면서** Precision/Recall 트레이드오프와 비용 최소화 임계값을 직접 계산하고, (4) PR-AUC가 "무조건 0" 모델과 어떤 관계를 갖는지(불균형 시 PR-AUC = 양성 비율) 확인합니다. numpy/scikit-learn만 씁니다.

## 1. 데이터: 스팸 필터 (양성 25% 불균형)

이메일 1개당 두 특징을 만든다: `x1` = 스팸 단어 빈도(0~100), `x2` = "발신자 미등록" 여부(0 또는 1). 정상 메일은 스팸 단어가 적고 대부분 발신자가 등록되어 있다 — 실제 스팸 데이터처럼 클래스가 겹치고, **정상 메일이 3배 많다.**

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
n_normal, n_spam = 400, 100   # 정상 80% / 스팸 20%

# 정상 메일: 스팸 단어 적음(평균 ~10개), 발신자 대부분 등록됨
x1n = rng.gamma(2, 5, n_normal)
x2n = (rng.random(n_normal) < 0.3).astype(float)
# 스팸의 70%는 "분명한" 스팸(스팸 단어 평균 ~36개), 30%는 정상 메일과 겹치는
# "교묘한" 스팸 — 그래서 임계값을 아무리 올려도 잡을 수 없는 스팸이 남는다
x1s = np.where(rng.random(n_spam) < 0.7, rng.gamma(2, 18, n_spam), rng.gamma(2, 5, n_spam))
x2s = (rng.random(n_spam) < 0.75).astype(float)

X = np.column_stack([np.r_[x1n, x1s], np.r_[x2n, x2s]])
y = np.r_[np.zeros(n_normal), np.ones(n_spam)].astype(int)
perm = rng.permutation(len(y)); X, y = X[perm], y[perm]

n_test = 150
X_test, y_test = X[:n_test], y[:n_test]
X_tr, y_tr = X[n_test:], y[n_test:]

# x1의 스케일(10~30)이 x2(0/1)의 30배라 학습률이 스케일에 맞춰야 한다(2.1 "학습률의 함정")
# — 그래서 스케일링 후 학습. 실전 로지스틱회귀 코드의 표준 전처리다.
from sklearn.preprocessing import StandardScaler
sc = StandardScaler().fit(X_tr)
X_test, X_tr = sc.transform(X_test), sc.transform(X_tr)

print(f"전체: {len(y)}개, 스팸 비율 = {y.mean():.2%}")
print(f"테스트: {len(y_test)}개, 스팸 {y_test.sum()}개 ({y_test.mean():.1%})")
print(f"학습  : {len(y_tr)}개, 스팸 {y_tr.sum()}개 ({y_tr.mean():.1%})")

전체: 500개, 스팸 비율 = 20.00%
테스트: 150개, 스팸 23개 (15.3%)
학습  : 350개, 스팸 77개 (22.0%)


## 2. 본문 `logistic_gradient_descent`로 학습 (그래디언트 \((h-y)\cdot x\))

책 본문의 순수 Python 구현을 numpy로 그대로 옮긴 것이다(비선형인 `h = sigmoid(w^Tx)`만 추가한 것 외, 2.1절 선형회귀 코드와 **구조가 동일**하다는 점을 확인해라).

In [2]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def logistic_gradient_descent(X, y, alpha=0.1, epochs=300):
    m, n = X.shape
    A = np.column_stack([np.ones(m), X])   # bias 열 추가 (x0=1 관례)
    w = np.zeros(n + 1)
    losses = []
    for epoch in range(epochs):
        h = sigmoid(A @ w)                 # <- 시그모이드만 추가된 부분
        losses.append(np.mean(-y * np.log(np.clip(h, 1e-12, 1-1e-12)) - (1 - y) * np.log(np.clip(1-h, 1e-12, 1-1e-12))))  # 교차 엔트로피
        grad = A.T @ (h - y) / m           # <- 2.1절과 똑같은 (h-y)·x
        w -= alpha * grad
    return w, losses

w, losses = logistic_gradient_descent(X_tr, y_tr)
print("bias, w1, w2 =", np.round(w, 3))
print(f"최종 교차 엔트로피 손실 = {losses[-1]:.4f}")

# sklearn의 로지스틱회귀(정규화 없는 버전)와 비교 — 거의 같아야 한다
from sklearn.linear_model import LogisticRegression
sk = LogisticRegression(C=1e6).fit(X_tr, y_tr)
print("sklearn ref   =", np.round(np.r_[sk.intercept_[0], sk.coef_[0]], 3))

bias, w1, w2 = [-1.494  1.345  0.608]
최종 교차 엔트로피 손실 = 0.3626
sklearn ref   = [-1.573  1.492  0.662]


## 3. 손으로 미분해 본 \(\sigma'(z)\) 확인: \(\sigma(z)(1-\sigma(z))\)

연습문제 5의 힌트 (2)를 수치로 검증한다. \(\sigma'(z)\)를 직접 미분한 것과 \(\sigma(z)(1-\sigma(z))\)가 모든 \(z\)에서 일치해야 하며, \(z=0\)에서 미분이 최대(0.25)임을 확인한다 — 시그모이드가 "가장 민감하게 반응하는 지점"이 정확히 50% 확률인 이유다.

In [3]:
z = np.linspace(-6, 6, 121)
numeric_deriv = np.gradient(sigmoid(z), z[1] - z[0])
analytic_deriv = sigmoid(z) * (1 - sigmoid(z))
print("max |수치미분 - 해석미분| =", f"{np.max(np.abs(numeric_deriv - analytic_deriv)):.2e}")
print("sigma'(0) =", sigmoid(0) * (1 - sigmoid(0)), " (최대치)")
assert np.max(np.abs(numeric_deriv - analytic_deriv)) < 1e-3

max |수치미분 - 해석미분| = 2.08e-04
sigma'(0) = 0.25  (최대치)


## 4. 임계값을 움직인다: Precision/Recall 트레이드오프

모델이 만든 **확률** \(p = h_w(x)\)을 임계값 0.5로만 자르지 말고, 0.05~0.95로 움직이면서 혼동행렬과 P/R/F1, 그리고 **비용**(FP=재검사 1건, FN=진단 지연 10건의 가중치)을 계산한다. 어떤 임계값이 비용을 최소화하는지 직접 찾아라 — "임계값은 하이퍼파라미터"라는 본문의 말이 숫자로 보인다.

In [4]:
def evaluate(y_true, prob, thr):
    pred = (prob >= thr).astype(int)
    TP = int(((pred == 1) & (y_true == 1)).sum())
    FP = int(((pred == 1) & (y_true == 0)).sum())
    FN = int(((pred == 0) & (y_true == 1)).sum())
    TN = int(((pred == 0) & (y_true == 0)).sum())
    P = TP / (TP + FP) if TP + FP else 0.0
    R = TP / (TP + FN) if TP + FN else 0.0
    F1 = 2 * P * R / (P + R) if P + R else 0.0
    acc = (TP + TN) / len(y_true)
    return TP, FP, FN, TN, P, R, F1, acc

probs = sigmoid(np.column_stack([np.ones(len(X_test)), X_test]) @ w)

print("thr    TP  FP  FN  TN   acc   P     R     F1    cost(FP+10·FN)")
for t in [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95]:
    TP, FP, FN, TN, P, R, F1, acc = evaluate(y_test, probs, t)
    print(f"{t:<5.2f} {TP:4d} {FP:4d} {FN:4d} {TN:4d}  {acc:5.3f} {P:5.3f} {R:5.3f} {F1:5.3f}  {FP + 10*FN}")

best = min((evaluate(y_test, probs, t) + (t,) for t in np.linspace(0.05, 0.95, 91)),
           key=lambda r: r[1] + 10 * r[2])
print(f"\n비용(FP=1, FN=10)을 최소화하는 임계값 ≈ {best[-1]:.2f}  (이때 FN={best[2]}, FP={best[1]})")

thr    TP  FP  FN  TN   acc   P     R     F1    cost(FP+10·FN)
0.05    21  119    2    8  0.193 0.150 0.913 0.258  139
0.10    19   62    4   65  0.560 0.235 0.826 0.365  102
0.20    16   24    7  103  0.793 0.400 0.696 0.508  94
0.30    12   14   11  113  0.833 0.462 0.522 0.490  124
0.40    11    7   12  120  0.873 0.611 0.478 0.537  127
0.50    10    2   13  125  0.900 0.833 0.435 0.571  132
0.60     9    0   14  127  0.907 1.000 0.391 0.562  140
0.70     7    0   16  127  0.893 1.000 0.304 0.467  160
0.80     4    0   19  127  0.873 1.000 0.174 0.296  190
0.90     4    0   19  127  0.873 1.000 0.174 0.296  190
0.95     2    0   21  127  0.860 1.000 0.087 0.160  210

비용(FP=1, FN=10)을 최소화하는 임계값 ≈ 0.14  (이때 FN=4, FP=44)


## 5. PR 곡선과 ROC 곡선 — 그리고 "무조건 0" 모델

PR-AUC와 ROC-AUC를 계산하고, **"무조건 0(정상)으로 예측하는 모델"** 을 넣는다. 정확도는 이 모델과 비슷해 보이지만(둘 다 80%대), **PR-AUC에서 이 모델은 정확히 양성 비율(0.20)을 받는다** — 본문의 정리 "PR-AUC가 무작위 모델과 같은 값이면 모델이 양성 클래스에 대해 아무 정보도 담고 있지 않다"의 직접 검증이다. ROC-AUC는 같은 무작위 모델에 0.5를 주기 때문에, 불균형 데이터에서는 PR-AUC가 더 "정직한" 기준선임을 보여준다.

In [5]:
from sklearn.metrics import average_precision_score, roc_auc_score, auc, precision_recall_curve

pr_auc = average_precision_score(y_test, probs)
roc_auc = roc_auc_score(y_test, probs)
prec, rec, _ = precision_recall_curve(y_test, probs)
print(f"모델의 PR-AUC = {pr_auc:.4f}, ROC-AUC = {roc_auc:.4f}")

# 무작위(무조건 0) 기준선
print(f"무조건 0 모델: 정확도 = {(y_test==0).mean():.3f}, PR-AUC = {average_precision_score(y_test, np.zeros_like(y_test)):.4f}, ROC-AUC = 0.5")
print(f"  -> PR-AUC 기준선은 정확히 양성 비율 {y_test.mean():.2f}과 일치 (본문 정리 확인)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]
ax.plot(rec, prec, lw=2, label=f"model (PR-AUC={pr_auc:.3f})")
ax.axhline(y_test.mean(), color="gray", ls="--", label=f"always-0 (PR-AUC={y_test.mean():.2f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve"); ax.legend(); ax.grid(alpha=0.3)

from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_test, probs)
ax = axes[1]
ax.plot(tpr, fpr, lw=2, label=f"model (ROC-AUC={roc_auc:.3f})")
ax.plot([0, 1], [0, 1], color="gray", ls="--", label="random (0.5)")
ax.set_xlabel("FPR (1 - Specificity)"); ax.set_ylabel("TPR (Recall)")
ax.set_title("ROC Curve"); ax.legend(); ax.grid(alpha=0.3)
fig.suptitle(f"Spam data: {int(y_test.sum())}/{len(y_test)} ({y_test.mean():.1%}) positives in test set")
fig.tight_layout()
fig.savefig("/tmp/ch02_3_pr_roc.png", dpi=120)
fig.savefig("/home/smhan/book-ml/kor/src/images/ch02_3_threshold_pr_curve.svg")
plt.show()

모델의 PR-AUC = 0.6417, ROC-AUC = 0.8049
무조건 0 모델: 정확도 = 0.847, PR-AUC = 0.1533, ROC-AUC = 0.5
  -> PR-AUC 기준선은 정확히 양성 비율 0.15과 일치 (본문 정리 확인)


## 6. 확인: `classification_report`와 `average_precision_score` (본문 실습 코드)

책 본문 "실습" 섹션의 코드를 그대로 실행하고, **같은 예측에 임계값 기반 지표와 확률 기반 지표가 얼마나 다른지** 비교한다. `y_pred`(임계값 0.5 고정)로 계산한 PR-AUC는 확률 `y_scores`로 계산한 PR-AUC보다 훨씬 낮다 — 임계값 하나에 갇힌 지표와 모델의 순서짓기 능력을 평가하는 지표의 차이가 수치로 보인다.

In [6]:
from sklearn.metrics import classification_report

y_true = [1, 1, 1, 0, 0, 0, 1, 0]
y_pred = [1, 0, 1, 0, 1, 0, 1, 0]
y_scores = [0.9, 0.4, 0.8, 0.3, 0.6, 0.2, 0.7, 0.1]

print(classification_report(y_true, y_pred, target_names=["정상", "스팸"]))
print("PR-AUC (scores):", round(average_precision_score(y_true, y_scores), 3))
print("PR-AUC (pred@0.5):", round(average_precision_score(y_true, y_pred), 3))
print("\n> 확률 기반 PR-AUC가 임계값 기반보다 높다는 점에 주목: 모델이 좋은 순서를 뽑고")
print("> 있더라도 임계값을 잘못 정하면 지표가 무너진다. 임계값은 성능 '진단' 이후에")
print("> 비용 함수로 선택할 것(섹션 4).")

              precision    recall  f1-score   support

          정상       0.75      0.75      0.75         4
          스팸       0.75      0.75      0.75         4

    accuracy                           0.75         8
   macro avg       0.75      0.75      0.75         8
weighted avg       0.75      0.75      0.75         8

PR-AUC (scores): 0.95
PR-AUC (pred@0.5): 0.688

> 확률 기반 PR-AUC가 임계값 기반보다 높다는 점에 주목: 모델이 좋은 순서를 뽑고
> 있더라도 임계값을 잘못 정하면 지표가 무너진다. 임계값은 성능 '진단' 이후에
> 비용 함수로 선택할 것(섹션 4).
